# Models

> Data models for the workflow session management interface

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

from cjm_workflow_state.state_store import SessionSummary

## Type Contracts

Callable types that hosts supply to customize session display without forcing the library to know about workflow-specific state schemas.

- `SessionEnricher`: turns raw `state_json` into display columns for the session list.
- `LabelGenerator`: derives a default human-readable label when a session's stored label is `None`.

In [ ]:
#| export
# Host-supplied: turn a raw state_json blob into a dict of display strings.
# Keys become column field names; values are already-formatted display strings.
SessionEnricher = Callable[[Dict[str, Any]], Dict[str, str]]

# Host-supplied: derive a human-readable label from a SessionSummary + raw state blob.
# Called only when summary.label is None.
LabelGenerator = Callable[[SessionSummary, Dict[str, Any]], str]

## EnrichedSessionSummary

Combines a raw `SessionSummary` from the state store with the resolved display label and any host-supplied enricher fields. This is what the UI actually renders.

In [ ]:
#| export
@dataclass
class EnrichedSessionSummary:
    """Session metadata plus resolved label and host-enriched display fields."""
    summary: SessionSummary # Raw session metadata from cjm-workflow-state
    resolved_label: str # summary.label if set, else label_generator output, else a default
    enriched_fields: Dict[str, str] = field(default_factory=dict) # Host enricher output — keyed by column field name

In [ ]:
# EnrichedSessionSummary round-trip with a stub SessionSummary.
_sum = SessionSummary(
    flow_id="flow1", session_id="sess1", label=None, current_step="selection",
    created_at="2026-04-08 10:00:00", updated_at="2026-04-08 11:00:00",
    state_size_bytes=512,
)
enriched = EnrichedSessionSummary(
    summary=_sum,
    resolved_label="Podcast Ep 42",
    enriched_fields={"sources": "3", "segments": "247"},
)
assert enriched.summary.session_id == "sess1"
assert enriched.resolved_label == "Podcast Ep 42"
assert enriched.enriched_fields["sources"] == "3"
print(f"EnrichedSessionSummary OK: {enriched.resolved_label} ({enriched.enriched_fields})")

EnrichedSessionSummary OK: Podcast Ep 42 ({'sources': '3', 'segments': '247'})


## ColumnSpec

Declares a single enricher-backed column for the session list — field name (key into `enriched_fields`) and display header.

In [ ]:
#| export
@dataclass
class ColumnSpec:
    """Declarative spec for a single enricher-backed column in the session list."""
    field: str # Key into EnrichedSessionSummary.enriched_fields
    header: str # Human-readable column header
    width_class: Optional[str] = None # Optional tailwind width class (e.g., "w-24") for the column

## SessionManagementUrls

URL bundle for all session-management routes. Constructed during router assembly and passed to components for link generation. Mirrors the `ManagementUrls` pattern from `cjm-transcript-workflow-management`.

In [ ]:
#| export
@dataclass
class SessionManagementUrls:
    """URL bundle for session management route endpoints."""
    management_page: str # GET: full session manager page
    list_sessions: str # GET: session list fragment (for OOB refresh)
    session_detail: str # GET: + ?session_id=... for detail view
    create_session: str # POST: mint a new session and switch to it
    delete_session: str # POST: + session_id in form data
    rename_session: str # POST: + session_id and new label
    resume_session: str # POST: set_session_id + redirect to workflow_url

## SessionManagementResult

Typed result dataclass returned by `init_session_manager_routers()`. Replaces a positional tuple with a named bundle for API clarity and non-breaking extensibility. Mirrors the `ManagementResult` pattern from `cjm-transcript-workflow-management`.

In [ ]:
#| export
@dataclass
class SessionManagementResult:
    """Result of session manager router initialization."""
    routers: List[Any] # APIRouter instances to register
    urls: SessionManagementUrls # URL bundle for route endpoints
    routes: Dict[str, Callable] # Route handler functions keyed by name
    render_page: Callable # () -> full session manager page component
    render_list: Callable # () -> session list component
    refresh_items: Callable # async () -> refresh items from the service
    resume_session: Callable[[Any, str], None] # (sess, session_id) -> set active session ID in the HTTP session

In [ ]:
_urls = SessionManagementUrls(
    management_page="/manage/sessions/",
    list_sessions="/manage/sessions/list",
    session_detail="/manage/sessions/detail",
    create_session="/manage/sessions/create",
    delete_session="/manage/sessions/delete",
    rename_session="/manage/sessions/rename",
    resume_session="/manage/sessions/resume",
)
_result = SessionManagementResult(
    routers=[], urls=_urls, routes={},
    render_page=lambda: None, render_list=lambda: None,
    refresh_items=lambda: None, resume_session=lambda sess, sid: None,
)
assert _result.urls.management_page == "/manage/sessions/"
print(f"SessionManagementResult OK: {len(_result.routers)} routers, urls={_result.urls.management_page}")

SessionManagementResult OK: 0 routers, urls=/manage/sessions/


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()